# 04 — Clustering + Skill-Gap (unsupervised)
KMeans on job vectors (elbow + silhouette). Skill-gap = cluster skills − resume skills. Topics optional.

In [1]:
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Locate the project folder
PROJECT_DIR = Path.cwd().parent

# Load the processed job dataset
jobs = pd.read_csv(
    PROJECT_DIR / "data" / "processed" / "jobs_clean.csv",
    usecols=lambda column: column in ["title", "company", "text"]
)

# Remove missing job descriptions
jobs = jobs.dropna(subset=["text"]).reset_index(drop=True)

print("Job dataset loaded successfully!")
print("Number of jobs:", len(jobs))
print("Available columns:", jobs.columns.tolist())
display(jobs.head())


Job dataset loaded successfully!
Number of jobs: 21346
Available columns: ['title', 'company', 'text']


,title,company,text
0,Walkin Data Entry Operator (night Shift),MM Media Pvt Ltd,walkin data entry operator night shift ites jo...
1,Work Based Onhome Based Part Time.,find live infotech,work based onhome based part time. marketing j...
2,Pl/sql Developer - SQL,Softtech Career Infosystem Pvt. Ltd,pl sql developer sql it software application p...
3,Manager/ad/partner - Indirect Tax - CA,Onboard HRServices LLP,manager ad partner indirect tax ca accounts jo...
4,JAVA Technical Lead (6-8 yrs) -,Spire Technologies and Solutions Pvt. Ltd.,java technical lead 6 8 yrs it software applic...


In [2]:
# Convert job descriptions into numerical features
tfidf = TfidfVectorizer(
    max_features=3000,
    stop_words="english"
)

job_matrix = tfidf.fit_transform(jobs["text"].astype(str))

print("TF-IDF features created successfully!")
print("Matrix shape:", job_matrix.shape)
print("Number of jobs:", job_matrix.shape[0])
print("Number of features:", job_matrix.shape[1])

TF-IDF features created successfully!
Matrix shape: (21346, 3000)
Number of jobs: 21346
Number of features: 3000


In [3]:
# Group similar jobs into 8 clusters
kmeans = KMeans(
    n_clusters=8,
    random_state=42,
    n_init=10
)

jobs["cluster"] = kmeans.fit_predict(job_matrix)

print("K-Means clustering completed successfully!")
print("\nNumber of clusters:", jobs["cluster"].nunique())
print("\nJobs in each cluster:")
print(jobs["cluster"].value_counts().sort_index())

print("\nSample jobs with their cluster labels:")
display(jobs[["title", "company", "cluster"]].head(10))

K-Means clustering completed successfully!

Number of clusters: 8

Jobs in each cluster:
cluster
0     820
1    5047
2    4328
3     842
4    5754
5    1149
6    2817
7     589
Name: count, dtype: int64

Sample jobs with their cluster labels:


,title,company,cluster
0,Walkin Data Entry Operator (night Shift),MM Media Pvt Ltd,4
1,Work Based Onhome Based Part Time.,find live infotech,4
2,Pl/sql Developer - SQL,Softtech Career Infosystem Pvt. Ltd,2
3,Manager/ad/partner - Indirect Tax - CA,Onboard HRServices LLP,3
4,JAVA Technical Lead (6-8 yrs) -,Spire Technologies and Solutions Pvt. Ltd.,2
5,WALK IN - As400 Developer - Pfsweb Global Serv...,PFS Web Global Services Pvt Ltd,1
6,PHP Developer,Kinesis Management Consultant Pvt. Ltd,2
7,Member Technical Staff-wire Harness/cable Harn...,Agile HR consultancy Pvt. Ltd. hiring for Ross...,1
8,Team Leader,HANSUM INDIA ELECTRONICS PVT.LTD.,4
9,German Translator,Accenture,1


In [4]:
# Evaluate clustering on a sample to reduce memory usage
from sklearn.metrics import silhouette_score

sample_size = min(1000, job_matrix.shape[0])

sample_indices = jobs.sample(
    n=sample_size,
    random_state=42
).index

sample_matrix = job_matrix[sample_indices]
sample_labels = jobs.loc[sample_indices, "cluster"]

score = silhouette_score(
    sample_matrix,
    sample_labels
)

print("Silhouette Score:", round(score, 4))

Silhouette Score: 0.0409
